In [1]:
import numpy as np
import pandas as pd 
from torchvision import datasets, transforms
import torch.nn as nn
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn as nn
from torch.utils.data import   DataLoader
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(
    root=r"fruits\dataset\train",
    transform=train_transform
)


test_dataset = datasets.ImageFolder(
    root=r"fruits\dataset\test",
    transform=test_transform
)




In [3]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [4]:
print(train_dataset.classes)
print(train_dataset.class_to_idx)

image, labels = next(iter(train_loader))



['Amaranth', 'Apple', 'Apricot', 'Asparagus', 'Avocado', 'Banana', 'Beans', 'Beetroot', 'Bell Pepper', 'Bitter Gourd', 'Black Berry', 'Black Current', 'Blueberry', 'Bottle Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cashew', 'Cauliflower', 'Chikoo', 'Chilli', 'Coconut', 'Corn', 'Cranberry', 'Cucumber', 'Custard Apple', 'Dates', 'Dragon Fruit', 'Elderberry', 'Fig', 'Garlic', 'Ginger', 'Gooseberry', 'Grapes', 'Guava', 'Jackfruit', 'Kiwi', 'Lemon', 'Lettuce', 'Litchi', 'Longan', 'Mango', 'Mushroom', 'Muskmelon', 'Okra', 'Olive', 'Onion', 'Orange', 'Papaya', 'Passion Fruit', 'Peach', 'Pear', 'Peas', 'Pineapple', 'Plum', 'Pomegranate', 'Potato', 'Pumpkin', 'Radish', 'Rambutan', 'Raspberry', 'Ridge Gourd', 'Spinach', 'Strawberry', 'Sweet Potato', 'Tamarind', 'Taro Roots', 'Tinda', 'Tomato', 'Turnip', 'Watermelon', 'Wax Gourd', 'Wood Apple', 'Yam', 'Zucchini']
{'Amaranth': 0, 'Apple': 1, 'Apricot': 2, 'Asparagus': 3, 'Avocado': 4, 'Banana': 5, 'Beans': 6, 'Beetroot': 7, '

In [5]:
# images, labels = next(iter(train_loader))

# print(images.shape)

In [13]:
# creating model
class my_cnn(nn.Module):
    def __init__(self,num_feature,num_classes):
        super().__init__()
        
        self.features=nn.Sequential(
            nn.Conv2d(num_feature,64,kernel_size=3,stride=2,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            
            nn.Conv2d(64,128,kernel_size=3,stride=2,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128,256,kernel_size=3,stride=2,padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            
            nn.AdaptiveAvgPool2d((1, 1))
            )
       
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self,x):
        output=self.features(x)
        output=self.classifier(output)
    
        return output
        

In [14]:
epochs=100
learning_rate=0.001
print(len(train_dataset.classes))
model=my_cnn(3, len(train_dataset.classes))
model=model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=learning_rate)

77


In [8]:
#training pipeline
for epoch in range(epochs):
    model.train()
    total_epoch_loss=0
    for image,labels in train_loader:
        image, labels = image.to(device), labels.to(device)
        
        output=model(image)
        
        
        loss=criterion(output, labels)
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss+=loss.item()
    avg_train_loss= total_epoch_loss/len(train_loader)
        
  
    
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Train Loss: {avg_train_loss:.5f} "
        )

Epoch [1/100] | Train Loss: 2.25225 
Epoch [2/100] | Train Loss: 1.60144 
Epoch [3/100] | Train Loss: 1.38050 
Epoch [4/100] | Train Loss: 1.23223 
Epoch [5/100] | Train Loss: 1.14101 
Epoch [6/100] | Train Loss: 1.04570 
Epoch [7/100] | Train Loss: 0.98526 
Epoch [8/100] | Train Loss: 0.92752 
Epoch [9/100] | Train Loss: 0.88838 
Epoch [10/100] | Train Loss: 0.83969 
Epoch [11/100] | Train Loss: 0.80369 
Epoch [12/100] | Train Loss: 0.78553 
Epoch [13/100] | Train Loss: 0.75076 
Epoch [14/100] | Train Loss: 0.71213 
Epoch [15/100] | Train Loss: 0.69632 
Epoch [16/100] | Train Loss: 0.68698 
Epoch [17/100] | Train Loss: 0.66604 
Epoch [18/100] | Train Loss: 0.63752 
Epoch [19/100] | Train Loss: 0.62555 
Epoch [20/100] | Train Loss: 0.61798 
Epoch [21/100] | Train Loss: 0.58965 
Epoch [22/100] | Train Loss: 0.57725 
Epoch [23/100] | Train Loss: 0.57169 
Epoch [24/100] | Train Loss: 0.56342 
Epoch [25/100] | Train Loss: 0.54987 
Epoch [26/100] | Train Loss: 0.54199 
Epoch [27/100] | Trai

In [9]:
# Model Evaluation

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_feature, batchlabel in train_loader:

        batch_feature = batch_feature.to(device)
        batchlabel = batchlabel.to(device)

        # model prediction
        output = model(batch_feature)

        # get predicted class
        _, predicted = torch.max(output, 1)

        # Count correct predictions
        total += batchlabel.size(0)
        correct += (predicted == batchlabel).sum().item()


accuracy = 100 * correct / total

print("MODEL PERFORMANCE CHECK")
print("Correct Predictions :", correct)
print("Total Images        :", total)
print(f"Accuracy            : {accuracy:.2f}%")

MODEL PERFORMANCE CHECK
Correct Predictions : 42137
Total Images        : 43324
Accuracy            : 97.26%


In [10]:
# Replace your Cell 10
# Model Evaluation
model.eval()

correct = 0
total = 0

with torch.no_grad():
    # FIXED: Iterate over test_loader instead of train_loader
    for batch_feature, batchlabel in test_loader: 

        batch_feature = batch_feature.to(device)
        batchlabel = batchlabel.to(device)

        # Model prediction
        output = model(batch_feature)

        # Get predicted class
        _, predicted = torch.max(output, 1)

        # Count correct predictions
        total += batchlabel.size(0)
        correct += (predicted == batchlabel).sum().item()

# Calculate accuracy
accuracy = 100 * correct / total

print("MODEL PERFORMANCE ON UNSEEN TEST DATA")
print("Correct Predictions :", correct)
print("Total Images        :", total)
print(f"Accuracy            : {accuracy:.2f}%")

MODEL PERFORMANCE ON UNSEEN TEST DATA
Correct Predictions : 3451
Total Images        : 3576
Accuracy            : 96.50%


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def predict_fruit(image_path, model, class_names, device):

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Same preprocessing used during training
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ])

    # Transform image
    image_tensor = transform(image)

    # Add batch dimension
    image_tensor = image_tensor.unsqueeze(0).to(device)

    # Prediction
    model.eval()

    with torch.no_grad():
        output = model(image_tensor)

        probabilities = torch.softmax(output, dim=1)

        predicted_class = torch.argmax(probabilities, dim=1).item()

        confidence = probabilities[0][predicted_class].item() * 100

    # Display image
    plt.imshow(image)
    plt.axis("off")
    plt.title(
        f"Prediction: {class_names[predicted_class]}\n"
        f"Confidence: {confidence:.2f}%"
    )
    plt.show()

    print("Predicted Fruit :", class_names[predicted_class])
    print(f"Confidence      : {confidence:.2f}%")
    
    

predict_fruit(
    "fruit.jpg",
    model,
    train_dataset.classes,
    device
)

In [6]:
#torch.save(model.state_dict(),"fruit_classification.pth")
import json
class_names=train_dataset.classes
with open("class_names.json", "w") as f:
    json.dump(class_names, f)